# Notebook 44: Memory Sync

This notebook explores distributed memory synchronisation patterns using `multigen.memory_sync`.
All examples use mock implementations — no real API keys are required.

Topics covered:
- `MVCCMemoryStore` — versioned writes with history and rollback
- Conflict resolution modes: `last_write_wins`, `first_write_wins`, `merge`
- `MemorySyncProtocol` — sync two independent stores
- `DistributedWorkingMemory` — multi-agent push with sliding window eviction
- `MemoryCoordinator` — multi-worker sync facade

In [ ]:
import sys
sys.path.insert(0, '../sdk')


## Mock Implementation of `multigen.memory_sync`

In [ ]:
import time
from copy import deepcopy
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

# ---------------------------------------------------------------------------
# 1. MVCCMemoryStore
# ---------------------------------------------------------------------------

@dataclass
class VersionedEntry:
    version: int
    value: Any
    timestamp: float = field(default_factory=time.time)


class MVCCMemoryStore:
    """Multi-Version Concurrency Control memory store."""

    def __init__(self, name: str = 'store'):
        self.name = name
        self._data: Dict[str, List[VersionedEntry]] = {}

    def write(self, key: str, value: Any) -> int:
        history = self._data.setdefault(key, [])
        version = len(history) + 1
        history.append(VersionedEntry(version=version, value=value))
        return version

    def read(self, key: str, version: Optional[int] = None) -> Any:
        history = self._data.get(key, [])
        if not history:
            raise KeyError(key)
        if version is None:
            return history[-1].value
        for entry in history:
            if entry.version == version:
                return entry.value
        raise KeyError(f'{key}@v{version}')

    def history(self, key: str) -> List[VersionedEntry]:
        return list(self._data.get(key, []))

    def rollback(self, key: str, to_version: int):
        history = self._data.get(key, [])
        self._data[key] = [e for e in history if e.version <= to_version]

    def keys(self) -> List[str]:
        return list(self._data.keys())

print('MVCCMemoryStore defined.')


## Section 1 — MVCCMemoryStore

In [ ]:
store = MVCCMemoryStore('main')

v1 = store.write('user_context', 'Hello, world!')
v2 = store.write('user_context', 'Hello, updated world!')
v3 = store.write('user_context', 'Final version.')

print(f'Current value (latest)  : {store.read("user_context")}')
print(f'Value at v1             : {store.read("user_context", version=1)}')
print(f'Value at v2             : {store.read("user_context", version=2)}')

print('\nFull history:')
for entry in store.history('user_context'):
    print(f'  v{entry.version}: {entry.value!r}')

# Rollback to v1
store.rollback('user_context', to_version=1)
print(f'\nAfter rollback to v1    : {store.read("user_context")}')
print(f'History length after rollback: {len(store.history("user_context"))}')


## Section 2 — Conflict Resolution

In [ ]:
def last_write_wins(existing: Any, incoming: Any) -> Any:
    return incoming

def first_write_wins(existing: Any, incoming: Any) -> Any:
    return existing

def merge_mode(existing: Any, incoming: Any) -> Any:
    if isinstance(existing, list) and isinstance(incoming, list):
        seen = set()
        merged = []
        for item in existing + incoming:
            key = str(item)
            if key not in seen:
                seen.add(key)
                merged.append(item)
        return merged
    if isinstance(existing, dict) and isinstance(incoming, dict):
        return {**existing, **incoming}
    # fallback: concatenate strings
    return str(existing) + ' | ' + str(incoming)


existing = ['fact_A', 'fact_B']
incoming = ['fact_B', 'fact_C']

print('Concurrent write scenario:')
print(f'  existing : {existing}')
print(f'  incoming : {incoming}')
print()
print(f'  last_write_wins  -> {last_write_wins(existing, incoming)}')
print(f'  first_write_wins -> {first_write_wins(existing, incoming)}')
print(f'  merge_mode       -> {merge_mode(existing, incoming)}')

# Dict merge
ex_dict = {'score': 0.8, 'label': 'positive'}
in_dict = {'score': 0.9, 'confidence': 0.95}
print(f'\n  merge_mode (dicts) -> {merge_mode(ex_dict, in_dict)}')


## Section 3 — MemorySyncProtocol

In [ ]:
@dataclass
class SyncResult:
    keys_synced: List[str]
    conflicts: List[str]


class MemorySyncProtocol:
    """Sync two MVCCMemoryStores; resolve conflicts with a chosen strategy."""

    def __init__(self, store_a: MVCCMemoryStore, store_b: MVCCMemoryStore,
                 conflict_strategy=last_write_wins):
        self.store_a = store_a
        self.store_b = store_b
        self.conflict_strategy = conflict_strategy

    def sync(self) -> SyncResult:
        keys_a = set(self.store_a.keys())
        keys_b = set(self.store_b.keys())
        all_keys = keys_a | keys_b

        synced = []
        conflicts = []

        for key in sorted(all_keys):
            in_a = key in keys_a
            in_b = key in keys_b

            if in_a and not in_b:
                self.store_b.write(key, self.store_a.read(key))
                synced.append(key)
            elif in_b and not in_a:
                self.store_a.write(key, self.store_b.read(key))
                synced.append(key)
            else:
                val_a = self.store_a.read(key)
                val_b = self.store_b.read(key)
                if val_a != val_b:
                    conflicts.append(key)
                    resolved = self.conflict_strategy(val_a, val_b)
                    self.store_a.write(key, resolved)
                    self.store_b.write(key, resolved)
                synced.append(key)

        return SyncResult(keys_synced=synced, conflicts=conflicts)


node1 = MVCCMemoryStore('node1')
node2 = MVCCMemoryStore('node2')

# Independent writes
node1.write('shared_key', 'value_from_node1')
node1.write('only_in_node1', 'exclusive')
node2.write('shared_key', 'value_from_node2')    # conflict!
node2.write('only_in_node2', 'also_exclusive')

print('Before sync:')
print(f'  node1 keys: {node1.keys()}')
print(f'  node2 keys: {node2.keys()}')

proto = MemorySyncProtocol(node1, node2, conflict_strategy=last_write_wins)
result = proto.sync()

print(f'\nSync result:')
print(f'  keys_synced : {result.keys_synced}')
print(f'  conflicts   : {result.conflicts}')
print(f'\nAfter sync:')
print(f'  node1["shared_key"] = {node1.read("shared_key")}')
print(f'  node2["shared_key"] = {node2.read("shared_key")}')
print(f'  node1 keys: {sorted(node1.keys())}')
print(f'  node2 keys: {sorted(node2.keys())}')


## Section 4 — DistributedWorkingMemory

In [ ]:
@dataclass
class MemoryEntry:
    agent_id: str
    content: str


class DistributedWorkingMemory:
    """Sliding-window working memory accepting pushes from multiple agents."""

    def __init__(self, max_window: int = 10):
        self.max_window = max_window
        self._entries: List[MemoryEntry] = []

    def push(self, agent_id: str, content: str):
        self._entries.append(MemoryEntry(agent_id=agent_id, content=content))
        # Evict oldest entries beyond window
        if len(self._entries) > self.max_window:
            evicted = self._entries.pop(0)
            print(f'  [evicted] agent={evicted.agent_id}: {evicted.content[:40]!r}')

    def context_text(self, separator: str = '\n') -> str:
        return separator.join(f'[{e.agent_id}] {e.content}' for e in self._entries)

    def __len__(self):
        return len(self._entries)


wm = DistributedWorkingMemory(max_window=5)

agents = ['agent_alpha', 'agent_beta', 'agent_gamma']
messages = [
    ('agent_alpha', 'Retrieved customer profile id=42'),
    ('agent_beta',  'Analysed sentiment: positive (0.87)'),
    ('agent_gamma', 'Fetched order history: 12 orders'),
    ('agent_alpha', 'Generated product recommendations'),
    ('agent_beta',  'Checked compliance rules: OK'),
    ('agent_gamma', 'Draft email prepared'),       # triggers eviction
    ('agent_alpha', 'Final response assembled'),   # triggers eviction
]

print('Pushing messages (window=5):\n')
for agent_id, content in messages:
    wm.push(agent_id, content)

print(f'\nWorking memory size: {len(wm)}')
print('\ncontext_text():')
print(wm.context_text())


## Section 5 — MemoryCoordinator

In [ ]:
class MemoryCoordinator:
    """Facade: register workers (MVCCMemoryStores), write/read, sync_all."""

    def __init__(self):
        self._workers: Dict[str, MVCCMemoryStore] = {}
        self._working_mem: Dict[str, DistributedWorkingMemory] = {}

    def register_worker(self, worker_id: str, window: int = 8):
        self._workers[worker_id] = MVCCMemoryStore(worker_id)
        self._working_mem[worker_id] = DistributedWorkingMemory(max_window=window)

    def write(self, worker_id: str, key: str, value: Any) -> int:
        return self._workers[worker_id].write(key, value)

    def read(self, worker_id: str, key: str) -> Any:
        return self._workers[worker_id].read(key)

    def push_working_memory(self, worker_id: str, content: str):
        self._working_mem[worker_id].push(worker_id, content)

    def working_memory_context(self, worker_id: str) -> str:
        return self._working_mem[worker_id].context_text()

    def sync_all(self, strategy=last_write_wins) -> Dict[str, SyncResult]:
        worker_ids = list(self._workers.keys())
        results = {}
        for i in range(len(worker_ids)):
            for j in range(i + 1, len(worker_ids)):
                id_a, id_b = worker_ids[i], worker_ids[j]
                proto = MemorySyncProtocol(self._workers[id_a], self._workers[id_b], strategy)
                results[f'{id_a}<->{id_b}'] = proto.sync()
        return results


coord = MemoryCoordinator()
for wid in ['worker_1', 'worker_2', 'worker_3']:
    coord.register_worker(wid)

# Workers write independently
coord.write('worker_1', 'task_result', 'output_A')
coord.write('worker_2', 'task_result', 'output_B')    # conflict with worker_1
coord.write('worker_3', 'task_result', 'output_A')    # matches worker_1
coord.write('worker_1', 'unique_key', 'only_w1')
coord.write('worker_3', 'another_key', 'only_w3')

# Each worker pushes to working memory
coord.push_working_memory('worker_1', 'Processed batch #1')
coord.push_working_memory('worker_2', 'Processed batch #2')
coord.push_working_memory('worker_3', 'Processed batch #3')

print('Before sync_all:')
for wid in ['worker_1', 'worker_2', 'worker_3']:
    store = coord._workers[wid]
    print(f'  {wid} keys: {store.keys()}')

print('\nRunning sync_all():')
sync_results = coord.sync_all(strategy=last_write_wins)
for pair, sr in sync_results.items():
    print(f'  {pair}: synced={sr.keys_synced}, conflicts={sr.conflicts}')

print('\nWorking memory per worker:')
for wid in ['worker_1', 'worker_2', 'worker_3']:
    print(f'  {wid}: {coord.working_memory_context(wid)}')
